In [ ]:
import io
from pdf2image import convert_from_path
from PIL import Image
import google.generativeai as genai
import os


class LocalMedicalPDFIngestor:
    def __init__(
        self,
        pdf_path: str,
        model_name: str = "gemini-2.5-flash"
    ):
        self.pdf_path = pdf_path
        self.model_name = model_name

        genai.configure(
            api_key=os.getenv("GEMINI_API_KEY")
        )

        self.model = genai.GenerativeModel(
            self.model_name
        )

    def _get_image_bytes(self, image: Image) -> bytes:
        """Converts a PIL Image to raw bytes."""
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG")
        return buffered.getvalue()

    def transcribe_page(
        self,
        image_bytes: bytes,
        page_num: int
    ) -> str:
        """Passes image bytes to Gemini."""

        prompt = """
        You are an expert medical transcriptionist.

        Read the following medical document page and extract ALL text exactly as written.

        - If there is handwriting, do your best to transcribe it.
        - If there is a table (like ICU charts, vitals, or labs), format it cleanly as a Markdown table.
        - Ignore generic anatomical diagrams (like the human body outlines for bed sores) UNLESS there are handwritten notes pointing to specific areas.
        - If there are notes, just transcribe the notes.
        - Do not add any conversational filler.

        Just return the extracted text.
        """

        try:

            response = self.model.generate_content(
                [
                    prompt,
                    {
                        "mime_type": "image/jpeg",
                        "data": image_bytes,
                    },
                ]
            )

            return response.text

        except Exception as e:
            print(
                f"Error calling Gemini on page {page_num}: {e}"
            )
            return (
                f"[ERROR TRANSCRIBING PAGE {page_num}]"
            )

    def process_pdf(self) -> str:
        """Converts PDF and compiles transcript."""

        print(
            f"Converting {self.pdf_path} to images..."
        )

        pages = convert_from_path(
            self.pdf_path
        )

        full_transcript = []

        for i, page in enumerate(pages):

            print(
                f"Transcribing page {i+1}/{len(pages)} using {self.model_name}..."
            )

            img_bytes = self._get_image_bytes(page)

            page_text = self.transcribe_page(
                img_bytes,
                i + 1
            )

            full_transcript.append(
                f"--- PAGE {i+1} ---\n{page_text}"
            )

        master_raw_text = "\n\n".join(
            full_transcript
        )

        return master_raw_text


# --- TEST EXECUTION ---

ingestor = LocalMedicalPDFIngestor(
    pdf_path="/home/sarthak/Downloads/patient.pdf",
    model_name="gemini-2.5-flash"
)

final_raw_text = ingestor.process_pdf()

print(final_raw_text[:1000])

/tmp/ipykernel_11466/1432282631.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Converting /home/sarthak/Downloads/patient.pdf to images...
